# PubMed Biomedical Metadata — Exploratory Analysis

A tour of the dataset: scale, coverage over time, journals, MeSH topics, author teams, abstracts, and how field completeness changes across the decades.

Run on the local clean corpus, abstract text is present and is analysed directly (length, structure, vocabulary). The same notebook also runs on the published Kaggle dataset, which is metadata-only — there the abstract-text sections detect the absence and skip gracefully, while every metadata section still works. A single flag set in the setup cell, `HAS_ABSTRACTS`, controls this.

## Setup

In [ ]:
import os, glob, re, collections
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# =====================================================================
# DATA LOADING — pick ONE option
# =====================================================================

# ---- OPTION A: LOCAL (active) -----------------------------------------
# Notebook runs from notebooks/, data sits one level up in the project root.
# Loads the full clean corpus (with abstracts); 2026 live-edge records are trimmed in 2b.
ROOT = os.path.dirname(os.getcwd())                       # go up one folder
DATA_DIR = os.path.join(ROOT, "data", "2_clean")          # full clean corpus (with abstracts)

# ---- OPTION B: KAGGLE (commented out — uncomment when running on Kaggle) ----
# # The published Kaggle data is metadata-only and already filtered to <= 2025, so 2b is a no-op.
# _hits = glob.glob("/kaggle/input/*/**/*.parquet", recursive=True) or glob.glob("/kaggle/input/*/*.parquet")
# assert _hits, "no parquet under /kaggle/input — attach the dataset as input"
# DATA_DIR = os.path.dirname(_hits[0])

# =====================================================================
df = pd.read_parquet(DATA_DIR)

# does this copy actually contain abstract text? (True locally, False on the metadata-only export)
HAS_ABSTRACTS = bool((df['abstract'].fillna('').str.len() > 0).any())

print(f"loaded {len(df):,} records from {DATA_DIR}")
print("abstract text present:", HAS_ABSTRACTS)
print("columns:", list(df.columns))
df.head(3)

In [ ]:
import sys
print("python :", sys.version.split()[0])
for m in ['pandas', 'numpy', 'matplotlib', 'seaborn']:
    print(f"{m:11}:", __import__(m).__version__)

## 1. Scale and schema
One row per article, keyed by PubMed ID (`uid`), with bibliographic metadata and flattened list fields (authors, affiliations, MeSH descriptors, keywords).

In [ ]:
print(f"records:       {len(df):,}")
print(f"unique PMIDs:  {df['uid'].nunique():,}")
print(f"year range:    {int(df['year'].min())}–{int(df['year'].max())}")
print(f"columns:       {df.shape[1]}")
df.dtypes

In [ ]:
# sanity checks: required columns present, and one row per PMID
required = ['uid', 'title', 'journal', 'year', 'pubdate_precision', 'n_authors',
            'n_mesh', 'n_keywords', 'has_coi', 'abstract', 'abstract_len',
            'mesh_descriptors', 'keywords', 'affiliations', 'source_month']
missing = [c for c in required if c not in df.columns]
assert not missing, f"missing expected columns: {missing}"
assert len(df) == df['uid'].nunique(), "duplicate PMIDs present — dedup did not run"
print("validation OK: all expected columns present, every row is a unique PMID")

## 2. Publications per year

### 2a. Volume trend
Annual volume grows steadily from the mid-1990s, accelerating through the 2010s, with a conspicuous dip around 2012–2015 (highlighted) and a small live-edge tail at the most recent year.

In [ ]:
per_year = df['year'].value_counts().sort_index()

plt.figure(figsize=(12, 4))
sns.lineplot(x=per_year.index, y=per_year.values, marker="o", color="#1d6fb8")
plt.axvspan(2012, 2015, color="#e07a5f", alpha=0.15)
plt.axvline(2012, color="#e07a5f", ls="--", lw=1.2)
plt.axvline(2015, color="#e07a5f", ls="--", lw=1.2)
seg = per_year.loc[2012:2015]
sns.lineplot(x=seg.index, y=seg.values, marker="o", color="#e07a5f", lw=2.5)
plt.title("Articles per year (2012–2015 dip highlighted)")
plt.xlabel("year"); plt.ylabel("articles")
plt.tight_layout(); plt.show()

### 2b. Trim the live edge (drop 2026)
The chart shows a tiny tail in 2026: a handful of electronic / ahead-of-print records plus still-accruing months — the unstable "live edge" of PubMed. They are dropped for a clean, frozen 1994–2025 view. This affects only the analysis; the underlying clean corpus is unchanged. (On Kaggle the data is already ≤ 2025, so this is a no-op.)

In [ ]:
MAX_YEAR = 2025
before = len(df)
df = df[df['year'] <= MAX_YEAR].copy()
print(f"trimmed to year <= {MAX_YEAR}: {len(df):,} records "
      f"(removed {before - len(df):,} live-edge rows)")
print(f"year range now: {int(df['year'].min())}–{int(df['year'].max())}")

## 2c. Diagnosing the 2012–2015 dip
Counts fall from a 2012 peak to a 2014 low (−32.5%), then recover — against otherwise steady 2–6%/year growth. The cells below establish the magnitude and isolate the cause.

### 2c-i. Magnitude (year-over-year change)

In [ ]:
per_year = df['year'].value_counts().sort_index()
stats = pd.DataFrame({"articles": per_year})
stats["yoy_change"] = stats["articles"].diff().astype("Int64")
stats["yoy_pct"] = (stats["articles"].pct_change() * 100).round(1)
print(stats.to_string())

### 2c-ii. By date precision
If the dip hit one date-format more than others it would point to a dating change. It does not — all three precision types fall together.

In [ ]:
prec_counts = df.groupby(['year', 'pubdate_precision']).size().unstack(fill_value=0)
plt.figure(figsize=(12, 4))
for col in prec_counts.columns:
    sns.lineplot(x=prec_counts.index, y=prec_counts[col], marker="o", label=col)
plt.axvspan(2012, 2015, color="#e07a5f", alpha=0.10)
plt.title("Article counts by date-precision, per year")
plt.xlabel("year"); plt.ylabel("articles"); plt.legend(title="precision")
plt.tight_layout(); plt.show()
print(prec_counts.loc[2011:2016].to_string())

### 2c-iii. By journal (the decisive test)
If the dip is uniform across journals it is systemic; if concentrated, specific sources drove it. Established US journals nearly vanish from the 2014 bucket while new megajournals grow — a wildly non-uniform ratio.

In [ ]:
top_j = df['journal'].value_counts().head(15).index
piv = df[df['journal'].isin(top_j)].groupby(['year', 'journal']).size().unstack(fill_value=0)
ratio = (piv.loc[2014] / piv.loc[2012]).sort_values()
print("2014/2012 article ratio for top-15 journals:")
print(ratio.round(2).to_string())
print(f"\nmedian ratio: {ratio.median():.2f}  "
      f"(uniform ~same = systemic; wide spread = specific journals drove it)")

**Implication:** the ratios split sharply rather than clustering around 1.0 (median 0.04). Established US journals nearly vanish from the 2014 bucket — Journal of Urology, Blood, Journal of Biological Chemistry, Cancer all ≈ 0.00x their 2012 count, PNAS 0.01x — while newly launched megajournals grow (Scientific Reports 4×, Nature Communications 6.5x). Flagship journals cannot have stopped publishing US research in 2014, so their collapse is the `USA[Affiliation]` filter failing to match their 2013–2014 records, not a real decline. The dip is therefore concentrated in established journals (an affiliation-indexing artifact), not a uniform systemic drop. This is the decisive evidence behind the §2c conclusion.

### 2c-iv. Affiliation coverage (circular — shown for completeness)
This cannot detect the dip: the query *requires* a US affiliation, so coverage is ~100% every year by construction. Included only to make that explicit.

In [ ]:
cov = df.assign(has_aff=df['affiliations'].map(lambda v: len(v) > 0)).groupby('year')['has_aff'].mean() * 100
print(cov.round(3).to_string())
print("\nNote: ~100% every year is expected — the query requires an affiliation, so this cannot detect the dip.")

**Conclusion — the dip is an affiliation-filter artifact, not a real decline.** All precision types drop ~35% together in 2014 (not a date-format issue). By journal the cause is clear: established US journals nearly vanish from the 2014 bucket (Journal of Biological Chemistry, Blood, Cancer ≈ 0.00x their 2012 count; PNAS 0.01x) while newly launched megajournals grow (Scientific Reports 4x, Nature Communications 6.5x). Those journals plainly did not stop publishing US research in 2014 — the `USA[Affiliation]` filter failed to match their 2013–2014 records, coinciding with PubMed's affiliation-indexing transition (~2013–2014, first-author-only → all-author affiliations). 
The harvest matches PubMed's own per-month counts exactly, so the dip is a faithful property of the filtered query — a metadata artifact, **not** a real change in output. Volume in 2013–2015 should be treated as a lower bound, not a trend.

## 3. Dates

### 3a. Date precision
Most records carry imprecise publication dates (year-only or year+month). Use `pubdate_precision` and restrict to `full_date` for month-level work.

In [ ]:
prec = df['pubdate_precision'].value_counts()
plt.figure(figsize=(7, 4))
sns.barplot(x=prec.index, y=prec.values, color="#1d6fb8")
plt.title("Publication-date precision"); plt.ylabel("records"); plt.xlabel("")
plt.tight_layout(); plt.show()
print((prec / len(df) * 100).round(1).astype(str) + " %")

**Which date field to use:**
- **`year`** — safe for all yearly trend analysis. Always populated.
- **`pubdate_precision`** — *always check this first* for any time-based work. It tells you how trustworthy the date is: `full_date` (day known), `year_month` (month known), `year` (only the year).
- **`pubdate`** — usable for month/day analysis **only when `pubdate_precision == 'full_date'`** (~23% of records). For the other ~77%, the month/day in `pubdate` is filled-in/approximate, not real — using it would invent precision the data doesn't have.
- **`source_month`** — the harvest bucket, *not* a publication date. Use it only for provenance/indexing-lag checks (§3b), never as the article's date.

Rule of thumb: yearly analysis → use `year` (all records). Monthly analysis → filter to `pubdate_precision == 'full_date'` first, and state that you did.

### 3a-ii. Are full-date days real?
`full_date` claims day-level precision, but PubMed often fills unknown days with "01." This checks the day-of-month distribution: if a large share land on the 1st (well above the ~3.3% a uniform spread would give), then many "full dates" are really month-level dates in disguise, and day-level analysis on them would be misleading.

In [ ]:
fd = df[df['pubdate_precision'] == 'full_date']
day = pd.to_datetime(fd['pubdate'], errors='coerce').dt.day
share_first = (day == 1).mean() * 100
print(f"full_date records: {len(fd):,}")
print(f"share landing on the 1st of the month: {share_first:.1f}%  (uniform ≈ 3.3%)")

plt.figure(figsize=(10, 3))
sns.histplot(day.dropna(), bins=31, color="#1d6fb8")
plt.title("Day-of-month distribution among full_date records")
plt.xlabel("day of month"); plt.ylabel("records"); plt.tight_layout(); plt.show()

**Reading it:** days are roughly uniform (~17k each) **except** large spikes on the 1st (~135k) and 15th (~92k). Those are default fill-ins — publishers assign the 1st or mid-month (15th) when the real day is unknown, even on records flagged `full_date`. So day-of-month is **not** reliable for these records: use `full_date` data at **month** granularity, and avoid any day-level analysis (e.g. day-of-week effects), which would be dominated by these artifacts.

### 3b. Indexing lag — gap between publication and harvest
Each record has a publication month (`pubdate`) and the month it was harvested (`source_month`). Their difference shows how long after publication a record appeared. A gap near zero means records show up right when they're published. Note: because the harvest was bucketed by publication date, this gap is expected to be near zero by design — it's a consistency check, not a measure of true indexing speed.

In [ ]:
fd = df[df['pubdate_precision'] == 'full_date']
if fd.empty:
    print("no full_date records — skipping.")
else:
    pm = pd.to_datetime(fd['pubdate'], errors='coerce').dt.to_period('M')
    sm = pd.PeriodIndex(fd['source_month'], freq='M')
    lag = (sm.astype('int64') - pm.astype('int64'))
    lag = lag[(lag >= -6) & (lag <= 24)]            # focus on the plausible window

    pct_same = (lag == 0).mean() * 100
    print(f"median gap: {int(lag.median())} months | mean: {lag.mean():.1f} months")
    print(f"harvested in the same month as published: {pct_same:.0f}%")

    plt.figure(figsize=(10, 3))
    sns.histplot(lag, bins=range(-6, 25), color="#8a5fb0")
    plt.axvline(0, color="black", ls="--", lw=1)     # mark "same month"
    plt.title("Months between publication and harvest")
    plt.xlabel("months (0 = same month)"); plt.ylabel("records")
    plt.tight_layout(); plt.show()

**Reading it:** 75% of records were harvested in the same month they were published (median gap 0, mean −0.6 — slightly negative from ahead-of-print records appearing early). Because the harvest was bucketed by publication date, this near-zero gap is **expected by construction** — it's a consistency check confirming the harvest is internally sound, not a measurement of PubMed's true indexing speed. The takeaway: no harvest-side lag inflates the recent-year counts.

## 4. Journals

### 4a. Top journals
The most prolific titles in the corpus.

In [ ]:
print("distinct journals:", df['journal'].nunique())
top = df['journal'].value_counts().head(15)[::-1]
plt.figure(figsize=(10, 6))
sns.barplot(x=top.values, y=top.index, color="#1d6fb8")
plt.title("Top 15 journals by article count"); plt.xlabel("articles"); plt.ylabel("")
plt.tight_layout(); plt.show()

### 4b. Concentration and new journals
How much of the corpus the top-N journals capture, and how many journals first appear each year. "First appearance" is within this corpus, not a journal's true founding year.

In [ ]:
total = len(df)
top_counts = df['journal'].value_counts()
print("journal concentration:")
for n in [5, 10, 20, 50, 100, 1000]:
    print(f"  top {n:>3} journals = {top_counts.head(n).sum()/total*100:5.1f}% of all articles")

first_seen = df.groupby('journal')['year'].min()
new_per_year = first_seen.value_counts().sort_index()
plt.figure(figsize=(12, 4))
sns.lineplot(x=new_per_year.index, y=new_per_year.values, marker="o", color="#2a9d5c")
plt.title("Journals by first appearance in the corpus (proxy, not founding year)")
plt.xlabel("year"); plt.ylabel("journals first seen"); plt.tight_layout(); plt.show()
print(new_per_year.tail(10).to_string())

**Reading it:** output is spread very widely — the top 5 journals are just 3.0% of articles, top 100 only 19.1%, across 7,316 journals. No handful of megajournals dominates; the corpus is a long tail of many modest contributors. (The Lorenz/Gini in 4e quantifies this — high Gini because thousands of journals have very few articles each, but no single journal is large.)

### 4c. Journal turnover — rank over time (bump chart)
How the leading journals rise and fall in rank. Established society journals lose ground while large open-access megajournals climb — the same turnover that drives the 2014 affiliation artifact in §2c.

Note: pre-2014 journal shares are affected by that same artifact — established journals are undercounted after ~2012 because their records stopped matching the US-affiliation filter, not because they stopped publishing. Read the turnover as "what the filtered corpus contains," not the real journal landscape.

In [ ]:
TOPN = 10
overall_top = df['journal'].value_counts().head(TOPN).index
counts = (df[df['journal'].isin(overall_top)]
          .groupby(['year', 'journal']).size().unstack(fill_value=0))
ranks = counts.rank(axis=1, ascending=False, method='min')

fig, axes = plt.subplots(2, 5, figsize=(16, 6), sharex=True, sharey=True)
for ax, j in zip(axes.flat, overall_top):
    ax.plot(ranks.index, ranks[j], marker='o', markersize=3, color="#1d6fb8")
    ax.invert_yaxis()
    ax.set_title(j[:25] + ("…" if len(j) > 25 else ""), fontsize=8)
    ax.set_yticks(range(1, TOPN + 1, 3))
fig.suptitle("Top journals — yearly rank (1 = most articles), one panel each", y=1.02)
fig.supxlabel("year"); fig.supylabel("rank")
plt.tight_layout(); plt.show()

### 4d. Journal share over time
The same turnover as a share of annual output, so the magnitude of each journal's rise or fall is visible (rank alone hides how big the gaps are).

In [ ]:
share = counts.div(df.groupby('year').size(), axis=0) * 100
share.plot.area(figsize=(13, 5), alpha=0.8)
plt.title("Top-10 journals — share of annual articles (%)")
plt.xlabel("year"); plt.ylabel("% of that year's articles")
plt.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8, frameon=False)
plt.tight_layout(); plt.show()
print("share %, selected years:")
print(share.loc[share.index.isin([1995, 2005, 2015, 2025])].round(2).to_string())

**Reading it:** the leaders completely turn over. In 1995 the top contributors were Journal of Biological Chemistry (1.67%), Scientific Reports (0.93% — actually near-zero then), and PNAS; by 2025 they are Scientific Reports (2.18%) and PNAS (1.91%), while JBC (0.33%) and Cancer Research (0.00%) have collapsed. 

Note the collapses overlap the affiliation artifact (§2c) — JBC and Cancer Research dropping to ~0 around 2015 is partly the filter, not purely a real decline — so read this as turnover *in the filtered corpus*, not a clean publishing-landscape measurement.

### 4e. Journal concentration — Lorenz curve and Gini
How unequally articles are distributed across all journals. The Lorenz curve plots cumulative share of articles against cumulative share of journals; Gini summarises it (0 = perfectly even, 1 = all articles in one journal).

In [ ]:
trapz = np.trapezoid if hasattr(np, "trapezoid") else np.trapz   # NumPy 2.x renamed trapz
jc = df['journal'].value_counts().sort_values().values
cum_articles = np.cumsum(jc) / jc.sum()
cum_journals = np.arange(1, len(jc) + 1) / len(jc)
gini = 1 - 2 * trapz(cum_articles, cum_journals)
half = int((cum_articles >= 0.5).argmax()); pct_journals = (len(jc) - half) / len(jc) * 100

plt.figure(figsize=(6, 6))
plt.plot(cum_journals, cum_articles, color="#1d6fb8", lw=2)
plt.plot([0, 1], [0, 1], ls="--", color="grey")
plt.title(f"Journal concentration (Gini = {gini:.2f})")
plt.xlabel("cumulative share of journals"); plt.ylabel("cumulative share of articles")
plt.tight_layout(); plt.show()
print(f"Gini = {gini:.2f}; the top {pct_journals:.0f}% of journals account for 50% of all articles")

**Reading it:** Gini = 0.73 — high inequality, as expected for academic publishing. The top 8% of journals carry 50% of all articles, while the remaining 92% share the other half (thousands with only a few articles each). This sits alongside the §4b finding (top 100 = 19%) without contradiction: no single journal is huge (so top-N shares stay low), but output is still very unevenly spread across the long tail (so Gini is high). The corpus is broad *and* unequal.

## 5. Authors

### 5a. Authors per paper
Right-skewed: most papers have a handful of authors, with a long tail of large collaborations.

In [ ]:
print(df['n_authors'].describe().astype(int))
plt.figure(figsize=(10, 4))
sns.histplot(df['n_authors'].clip(upper=30), bins=30, color="#1d6fb8")
plt.title("Authors per paper (clipped at 30)"); plt.xlabel("authors"); plt.ylabel("papers")
plt.tight_layout(); plt.show()

**Reading it:** the median paper has 5 authors and the middle 50% fall between 3 and 7. The mean (5) sits at the median here, but the max of 2,929 shows a thin tail of huge consortia. That tail is real (large multi-site collaborations) but distorts any per-paper average and will dominate co-authorship-network centrality — so exclude or handle 21+ author papers separately in those analyses.

### 5b. Team size over time
The composition shifts toward larger teams — solo work shrinks, large collaborations grow.

In [ ]:
band = pd.cut(df['n_authors'], [0, 1, 5, 20, 10**9], labels=["solo", "2–5", "6–20", "21+"])
collab = df.assign(band=band).groupby(['year', 'band'], observed=True).size().unstack(fill_value=0)
collab_pct = collab.div(collab.sum(axis=1), axis=0) * 100

collab_pct.plot.area(figsize=(12, 4), color=["#d9534f", "#1d6fb8", "#2a9d5c", "#8a5fb0"])
plt.title("Team-size composition by year (%)"); plt.xlabel("year"); plt.ylabel("% of articles")
plt.legend(title="authors", loc="lower left"); plt.tight_layout(); plt.show()
print("share by band, selected years:")
print(collab_pct.loc[collab_pct.index.isin([1995, 2005, 2015, 2025])].round(1).to_string())

**Reading it:** a clear shift toward team science across 30 years. Solo authorship collapses (14.6% → 2.5%), the small-team 2–5 band shrinks (64.6% → 36.3%), and large teams grow — the 6–20 band goes from 20.8% to 55.3% (now the majority) and 21+ author papers from 0.0% to 5.9%. Summarized by the effect-size line: papers with 6+ authors rose from 23% (1994–1999) to 58% (2020–2025). This is a genuine, well-documented trend in science, not an artifact — and it sets up the co-authorship-network analysis (restrict to post-2014 for reliable affiliations).

In [ ]:
# effect size (not a p-value: at ~3M rows every test is "significant" and uninformative)
early = (df[df['year'] <= 1999]['n_authors'] >= 6).mean() * 100
late  = (df[df['year'] >= 2020]['n_authors'] >= 6).mean() * 100
print(f"share of papers with 6+ authors: {early:.0f}% (1994–1999) -> {late:.0f}% (2020–2025)")

### 5c. Correlation among numeric features
How the numeric fields relate. Correlations are expected to be weak; these are descriptive, not inferential (no multiple-comparison correction). Any correlation involving `year` is confounded by the indexing artifacts above (the 2014 dip, the MeSH-depth shift, the keyword/COI onset) — read `year` rows with caution.

In [ ]:
num_cols = ['n_authors', 'n_mesh', 'n_keywords', 'abstract_len', 'year']
corr = df[num_cols].corr()
plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, fmt='.2f')
plt.title('Correlation among numeric features (descriptive)'); plt.tight_layout(); plt.show()

**Reading it:** correlations among the numeric fields are weak overall (most |r| < 0.25), which is expected — these features are loosely coupled. Two cells need careful interpretation rather than face-value reading:

- **`n_keywords` ↔ `year` = 0.60** looks strong but is an **artifact, not a relationship**: keywords were near-absent before ~2012 and common after, so the keyword *count* rises with year purely because the field was introduced over time — not because newer papers "have more keywords" in any meaningful sense.
- **`n_mesh` ↔ `year` = −0.21** is the MeSH-depth drop (§6c): the post-2020 automated-indexing change lowered descriptors per article, so MeSH count falls with year. Again a process change, not a content one.

The genuine (if weak) signals: more authors loosely tracks more MeSH terms and longer abstracts (`n_authors` ↔ `n_mesh` 0.12, ↔ `abstract_len` 0.19) — bigger studies are slightly richer. These are descriptive, not inferential (no significance testing — at 3M rows every coefficient would be "significant" and that would mean nothing).

## 6. MeSH topics

### 6a. Top descriptors
Most frequent MeSH descriptors — dominated by the human-subject scope of the corpus.

In [ ]:
mc = collections.Counter(d for lst in df['mesh_descriptors'] for d in lst)
print("distinct MeSH descriptors:", len(mc))
top = pd.Series(dict(mc.most_common(15)))[::-1]
plt.figure(figsize=(10, 6))
sns.barplot(x=top.values, y=top.index, color="#2a9d5c")
plt.title("Top 15 MeSH descriptors"); plt.xlabel("occurrences"); plt.ylabel("")
plt.tight_layout(); plt.show()

**What this shows:** the most frequent descriptors are all demographic/structural, not topical — "Humans", "Female", "Male", "Adult", "Middle Aged", "Aged", "Animals". This is expected: PubMed tags nearly every human-subject paper with these, so they dominate by sheer ubiquity. The practical consequence: these generic descriptors carry almost no topical signal and must be removed before any clustering, co-occurrence, or similarity analysis — otherwise every paper looks similar because they all share "Humans"/"Female"/"Male".

### 6b. Top descriptors over time (raw counts)
Raw counts of the leading descriptors per year. These largely track total volume — see 6e for the volume-normalized version.

In [ ]:
TOP_N = 15
top_descriptors = [d for d, _ in mc.most_common(TOP_N)]
mesh_exploded = (df[["year", "mesh_descriptors"]]
                 .explode("mesh_descriptors")
                 .rename(columns={"mesh_descriptors": "descriptor"}))
mesh_exploded = mesh_exploded[mesh_exploded["descriptor"].isin(top_descriptors)]
mesh_ts = mesh_exploded.groupby(["year", "descriptor"]).size().reset_index(name="count")

fig, ax = plt.subplots(figsize=(13, 6))
for desc in top_descriptors:
    sub = mesh_ts[mesh_ts["descriptor"] == desc]
    ax.plot(sub["year"], sub["count"], label=desc, linewidth=1.5)
ax.set_title("Top 15 MeSH descriptors — raw article count per year")
ax.set_xlabel("year"); ax.set_ylabel("articles")
ax.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8, frameon=False)
plt.tight_layout(); plt.show()

**What this shows:** every descriptor's raw count rises and falls together, tracking total article volume — including the 2014 dip and the 2020–2021 surge. That co-movement is the point: raw counts mostly measure *how many articles there were*, not whether a topic grew in importance. "Humans" tracks the overall volume curve almost exactly. The sharp 2022–2023 dip then 2024–2025 rebound visible here is the MeSH-indexing change (fewer descriptors assigned, §6c), not a topical shift. Because of this confounding, the volume-normalized view (§6e) is the one to read for genuine prevalence changes.

### 6c. Descriptors per article over time
The structural shift: descriptor depth is stable until ~2019, then drops.

In [ ]:
mesh_year = df.groupby('year')['n_mesh'].mean()
plt.figure(figsize=(12, 4))
sns.lineplot(x=mesh_year.index, y=mesh_year.values, marker="o", color="#2a9d5c")
plt.title("Mean MeSH descriptors per article, by year"); plt.xlabel("year"); plt.ylabel("mean # MeSH")
plt.tight_layout(); plt.show()
print(mesh_year.round(2).to_string())

**What this shows:** descriptor depth holds steady around 13 from 1994–2019 (12.4–13.4), then drops sharply — 11.55 (2020), 9.96 (2021), 8.24 (2022), 8.09 (2023) — before partially rebounding to ~11 in 2024–2025. The cliff is far too abrupt to reflect any change in research content; it tracks NLM's 2021–2022 move to automated MeSH indexing, which assigns fewer descriptors per article, plus indexing lag on recent years that 2024–25 are still catching up on. Treat MeSH depth as era-dependent: pre-2020 and post-2020 records are not directly comparable, so segment by era for any breadth/co-occurrence analysis.

### 6d. Volume vs MeSH depth side by side
Total articles and mean descriptors per article, to confirm the depth decline is independent of volume.

In [ ]:
mesh_coverage = df.groupby("year").agg(
    total=("uid", "count"),
    with_mesh=("n_mesh", lambda x: (x > 0).sum()),
    avg_mesh=("n_mesh", "mean")
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(mesh_coverage["year"], mesh_coverage["total"], color="#1d6fb8")
axes[0].set_title("total articles per year"); axes[0].set_xlabel("year")
axes[1].plot(mesh_coverage["year"], mesh_coverage["avg_mesh"], color="#e0853f")
axes[1].set_title("avg MeSH descriptors per article"); axes[1].set_xlabel("year")
plt.tight_layout(); plt.show()

**What this shows:** placing the two curves together separates the effects. Total volume (left) has the 2014 affiliation dip and the 2020–21 surge; MeSH depth (right) is flat until 2019 then collapses. They move *independently* — the depth drop is not caused by the volume changes — which confirms the depth decline is an indexing-process change (§6c), not a side-effect of how many articles were published.

### 6e. Top descriptors normalized per 1,000 articles
Dividing by articles per year shows which topics genuinely rose or fell in *prevalence*, independent of corpus growth.

In [ ]:
art_per_year = df.groupby('year').size()
mex = df[['year', 'mesh_descriptors']].explode('mesh_descriptors')
top_norm = mex['mesh_descriptors'].value_counts().head(10).index
norm = (mex[mex['mesh_descriptors'].isin(top_norm)]
        .groupby(['year', 'mesh_descriptors']).size().unstack(fill_value=0)
        .div(art_per_year, axis=0) * 1000)

plt.figure(figsize=(13, 6))
for d in top_norm:
    plt.plot(norm.index, norm[d], label=d, lw=1.5)
plt.title("Top MeSH descriptors — prevalence per 1,000 articles per year")
plt.xlabel("year"); plt.ylabel("per 1,000 articles")
plt.legend(bbox_to_anchor=(1.01, 1), loc="upper left", fontsize=8, frameon=False)
plt.tight_layout(); plt.show()

**What this shows:** dividing by articles-per-year removes the volume confound, so this reflects genuine prevalence. "Humans" sits flat at ~1,000 per 1,000 — i.e. ~100% of records, by construction (the query filters to human subjects), so it carries no information and can be ignored. The informative lines are below it: "Female" (~360→560 per 1,000) and "Male" rise gradually, reflecting better demographic indexing over time, while most others stay broadly flat. The shared dip at 2022–2023 then rebound is again the indexing-depth change (§6c) showing through — even normalized, fewer descriptors per article means every term's per-1,000 rate dips in those years. So read relative *changes* between terms, not the absolute 2022–23 trough.

MeSH coverage is 100% across all years — every record carries at least one descriptor (PubMed releases fully-indexed records). But the **mean number of descriptors per article is stable at ~13 through 2019, then falls to ~8 by 2022–2023**, with a partial rebound in 2024–2025. This matches NLM's transition to automated indexing (~2021–2022, fewer descriptors) plus indexing lag on recent years — not a content change. MeSH-breadth analyses should segment by era and treat post-2019 depth cautiously.

## 7. Field completeness over time

### 7a. Coverage trend (with auto-detected thresholds)
Keywords (from ~2012) and conflict-of-interest statements (from ~2017) became standard only gradually. The "valid from" thresholds are computed from the data rather than hardcoded, so they stay correct if the corpus updates.

In [ ]:
by_year = df.assign(
    has_kw=df['n_keywords'] > 0,
    has_mesh=df['n_mesh'] > 0,
).groupby('year').agg(
    keywords=('has_kw', 'mean'),
    coi=('has_coi', 'mean'),
    mesh=('has_mesh', 'mean'),
) * 100

plt.figure(figsize=(12, 4))
sns.lineplot(data=by_year, dashes=False, markers=False)
plt.title("Field coverage by year (%)"); plt.xlabel("year"); plt.ylabel("% of records")
plt.legend(title=""); plt.tight_layout(); plt.show()

def first_year_at(series, thresh, sustain=3):
    """First year coverage reaches `thresh`% and stays >= thresh for `sustain` consecutive years.
    The `sustain` requirement avoids a one-off early blip being reported as the onset."""
    s = series.sort_index()
    for i in range(len(s) - sustain + 1):
        if (s.iloc[i:i + sustain] >= thresh).all():
            return int(s.index[i])
    return None

print(f"keywords first reach 50% coverage: {first_year_at(by_year['keywords'], 50)}")
print(f"COI statements first reach 5% coverage: {first_year_at(by_year['coi'], 5)}")

**What this shows:** the three fields have completely different histories. MeSH is flat at 100% throughout — every record is fully indexed, which is how PubMed releases them. Keywords are near-zero until ~2012, then climb steeply to ~80% by 2020 (author keywords became a standard submission field only in the 2010s). COI statements start later and rise more slowly, reaching ~75% by 2025. The crossover years are computed with a sustained-threshold rule (the value must hold for three consecutive years) so a one-off early blip is not mistaken for the real onset. Practical rule: keyword analysis is valid only from ~2013 onward, COI analysis only from the late 2010s.

### 7b. Completeness heatmap
The same story as a year x field grid. "Missing" means the field is **empty** (empty list / False / zero) — these columns are never null in this schema. Darker = more missing.

In [ ]:
present = pd.DataFrame({
    "keywords":     df['n_keywords'] > 0,
    "COI":          df['has_coi'],
    "affiliations": df['affiliations'].map(lambda v: len(v) > 0),
})
missing_by_year = (1 - present.groupby(df['year']).mean()) * 100
plt.figure(figsize=(12, 4))
sns.heatmap(missing_by_year.T, annot=True, fmt='.0f', cmap='YlOrRd',
            cbar_kws={'label': '% missing (empty)'})
plt.title('Missing (empty) data by year, sparse fields'); plt.tight_layout(); plt.show()

**What this shows:** the same story as a year x field grid (darker = more missing). Affiliations are 0% missing every year — the query requires one, so it is present by construction. Keywords are ~96–100% missing through 2012, then fall fast: 82% (2013), 52% (2014), down to ~17% by 2025. COI is 100% missing until ~2005, stays mostly empty through the early 2010s, and only drops below half after ~2019 (54% in 2020 → 25% by 2025). This grid is the authority for the "valid from" cutoffs used elsewhere in the notebook.

### 7c. COI statement classification (local corpus only)
A rule-based split of the free-text COI statement. Papers with **no COI field at all** ("no statement") are kept separate from those that **state no conflict** ("states none") — a distinction that matters for any transparency analysis. Needs `coi_statement` text, present only in the local clean corpus; on the metadata-only export the cell skips. The keyword rules are heuristic; treat as indicative, not exact.

In [ ]:
if df['coi_statement'].fillna('').str.len().gt(0).any():
    def classify_coi(text):
        t = (text or "").lower()
        if any(p in t for p in ['no conflict', 'no competing', 'declare no', 'none declared', 'nothing to disclose']):
            return 'states none'
        if any(p in t for p in ['received', 'grants', 'consultant', 'advisory board', 'speaker', 'honoraria', 'stock', 'employment', 'paid']):
            return 'discloses conflict'
        return 'ambiguous'
    # has_coi separates "no field at all" from statements that exist
    df['coi_class'] = np.where(~df['has_coi'], 'no statement',
                               df['coi_statement'].apply(classify_coi))
    print(df['coi_class'].value_counts().to_string())
    coi_ts = df.groupby('year')['coi_class'].value_counts(normalize=True).unstack().fillna(0) * 100
    coi_ts.plot.area(figsize=(12, 4), alpha=0.7)
    plt.title('COI statement classification over time (%)'); plt.xlabel('year'); plt.ylabel('%')
    plt.legend(bbox_to_anchor=(1.01, 1), loc='upper left'); plt.tight_layout(); plt.show()
else:
    print("coi_statement is empty (metadata-only export) — skipping; run on the local clean corpus.")

**What this shows:** decomposing the COI statements ("no statement" = no COI field at all, separated from statements that exist via `has_coi`). Until ~2005 essentially everything is "no statement" — the field did not exist. From the late 2010s it shrinks as statements appear: "states none" grows to the largest disclosed share, "discloses conflict" stays a small but real slice (~a few %), and "ambiguous" rises in recent years. Read as transparency: among papers that carry any statement, most declare no conflict and roughly 1 in 10 disclose one — but the dominant signal is the *introduction* of the field over time, not a change in real conflict rates. The growing "ambiguous" band flags that the rule-based classifier misses nuanced wording, so treat the breakdown as indicative, not exact.

## 8. Text fields (titles and abstracts)


A **metadata-level** look at the text: lengths, structure, and basic vocabulary. This characterises the text and confirms it is usable, but does **not** analyse content — tokenization, named-entity recognition, topic modelling, and sentiment are handled in dedicated downstream notebooks (e.g. `04_topic_modelling`, the tokenization/NER notebook). The goal here is only to size and sanity-check the text before that deeper work.

### 8a. Abstract length distribution
The original abstract length (`abstract_len`) is retained even in the metadata-only export, so the distribution can always be shown.

In [ ]:
print(f"mean original abstract length: {df['abstract_len'].mean():.0f} chars")
plt.figure(figsize=(10, 3))
sns.histplot(df['abstract_len'].clip(upper=4000), bins=50, color="#8a5fb0")
plt.title("Original abstract length (chars, clipped 4000)"); plt.xlabel("chars")
plt.tight_layout(); plt.show()

**What this shows:** abstract length (mean ~1,419 chars) is roughly bell-shaped with a right tail of long abstracts. This is a structural overview only — the actual text content (entities, topics, sentiment) is not analysed here. Deeper text work is deferred to dedicated notebooks: tokenization and named-entity recognition, and topic modelling over the abstract corpus.

### 8b. Title length, and abstract length over time
Titles are always present (the only text in the published release). Abstracts have lengthened steadily over the decades.

In [ ]:
tl = df['title'].fillna('').str.len()
print(f"title length — mean {tl.mean():.0f}, median {int(tl.median())}, "
      f"min {int(tl.min())}, max {int(tl.max())} chars")
plt.figure(figsize=(10, 3))
sns.histplot(tl.clip(upper=300), bins=60, color="#1d6fb8")
plt.title("Title length (chars)"); plt.xlabel("chars"); plt.tight_layout(); plt.show()

abs_year = df.groupby('year')['abstract_len'].mean()
plt.figure(figsize=(12, 4))
sns.lineplot(x=abs_year.index, y=abs_year.values, marker="o", color="#8a5fb0")
plt.title("Mean abstract length over time (chars)"); plt.xlabel("year"); plt.ylabel("mean chars")
plt.tight_layout(); plt.show()
print(abs_year.round(0).astype(int).to_string())

**What this shows:** titles are always present (mean ~99 chars) and abstracts have lengthened steadily — ~1,134 chars (1994) to ~1,639 (2025), a ~45% increase with no reversals, reflecting a long-run norm shift toward fuller abstracts. Length is treated here purely as metadata; the words themselves are left for the later text-mining notebooks, where this length trend matters for chunking and model input limits.

### 8c. Team size vs abstract length
Tests whether larger teams write longer abstracts. Both quantities are heavy-tailed (a few papers have hundreds of authors or very long abstracts), which would squash a linear plot into one corner; plotting on log–log spreads the dense range so the relationship is visible, and the fit then estimates a power-law rather than additive trend. Expect a weak positive relationship at most.

In [ ]:
s = df[(df['n_authors'] > 0) & (df['abstract_len'] > 0)].sample(min(10000, len(df)), random_state=42)
plt.figure(figsize=(8, 6))
sns.regplot(x=np.log10(s['n_authors']), y=np.log10(s['abstract_len']),
            scatter_kws={'alpha': 0.3, 's': 5}, line_kws={'color': 'red'})
plt.xlabel('log10(authors)'); plt.ylabel('log10(abstract length)')
plt.title('Authors vs abstract length (log–log)'); plt.tight_layout(); plt.show()
print("Pearson r (log-log):",
      round(np.corrcoef(np.log10(s['n_authors']), np.log10(s['abstract_len']))[0, 1], 3))

**What this shows:** a weak positive relationship at most — larger teams write only marginally longer abstracts (the log–log fit is nearly flat, Pearson r printed above is small). Abstract length is driven by norms and journal requirements, not team size. This closes the metadata-level look at abstracts; semantic relationships between text and other fields belong to the embedding/topic-modelling work in later notebooks.

### 8d. Abstract length by era and team size
Whether abstracts lengthened uniformly or only in certain team-size groups. Median characters by era × author band (uses `abstract_len`, retained in the export, so this runs everywhere).

In [ ]:
band = pd.cut(df['n_authors'], [0, 1, 5, 20, 10**9], labels=["solo", "2–5", "6–20", "21+"])
era = pd.cut(df['year'], [0, 2009, 2019, 2100], labels=["pre-2010", "2010–19", "2020+"])
strat = (df.assign(band=band, era=era)
           .groupby(['era', 'band'], observed=True)['abstract_len'].median().unstack())
print(strat.round(0).astype('Int64').to_string())
strat.T.plot(kind='bar', figsize=(10, 4))
plt.title("Median abstract length (chars) by era and team size")
plt.ylabel("median chars"); plt.xlabel("team size"); plt.legend(title="era")
plt.tight_layout(); plt.show()

**What this shows:** abstract lengthening is broadly uniform across team-size bands and eras rather than driven by one group — the rise is a general trend, not a large-team effect. This is the last metadata-only view of the text fields; from here, analysis that uses the abstract *content* moves to the dedicated downstream notebooks.

### 8e. Abstract word and sentence counts (abstract text required)
Word and sentence counts per abstract, and mean length-in-words over time. Requires abstract text — present locally, skipped on the metadata-only export.

In [ ]:
if HAS_ABSTRACTS:
    wc = df['abstract'].fillna('').str.split().str.len()
    sc = df['abstract'].fillna('').str.count(r'[.!?]')
    print(f"words per abstract — mean {wc.mean():.0f}, median {int(wc.median())}")
    print(f"sentences per abstract — mean {sc.mean():.0f}, median {int(sc.median())}")
    wc_year = df.assign(wc=wc).groupby('year')['wc'].mean()
    plt.figure(figsize=(12, 4))
    sns.lineplot(x=wc_year.index, y=wc_year.values, marker="o", color="#8a5fb0")
    plt.title("Mean abstract length over time (words)"); plt.xlabel("year"); plt.ylabel("mean words")
    plt.tight_layout(); plt.show()
else:
    print("abstract text absent (metadata-only export) — skipping word/sentence analysis.")

**What this shows:** when abstract text is present (local corpus), basic counts — words and sentences per abstract, and mean words over time. This is a lightweight first pass to confirm the text is well-formed and to size it; full tokenization, lemmatization, and cleaning are handled properly in the tokenization/NER notebook rather than here.

### 8f. Structured vs unstructured abstracts (abstract text required)
Many modern abstracts use explicit section labels (Background / Methods / Results / Conclusions). The share that are structured rises over time — useful for any downstream parsing of abstract sections. Label detection is heuristic (presence of the words), so read it as a trend rather than an exact count.

In [ ]:
if HAS_ABSTRACTS:
    labels = ['background', 'methods', 'results', 'conclusion', 'conclusions',
              'objective', 'objectives', 'introduction', 'purpose', 'findings']
    pat = re.compile(r'\b(?:' + '|'.join(labels) + r')\b', re.I)
    df['is_structured'] = df['abstract'].fillna('').str.contains(pat)
    struct_year = df.groupby('year')['is_structured'].mean() * 100
    print(f"overall structured: {df['is_structured'].mean()*100:.0f}%")
    plt.figure(figsize=(12, 4))
    sns.lineplot(x=struct_year.index, y=struct_year.values, marker="o", color="#2a9d5c")
    plt.title("Share of abstracts with section labels, by year (%)")
    plt.xlabel("year"); plt.ylabel("% structured"); plt.tight_layout(); plt.show()
else:
    print("abstract text absent (metadata-only export) — skipping structure detection.")

**What this shows:** the share of abstracts carrying explicit section labels (Background / Methods / Results / Conclusions) rises over time, so newer abstracts are increasingly parseable into sections. Detection here is a simple keyword heuristic — a proper structured-abstract parser (splitting each section into its own field) is left for the text-processing notebook, where this trend tells you which records can be sectioned reliably.

### 8g. Most common abstract words (abstract text required)
A quick vocabulary view with a small stop-word filter — a rough sense of dominant terms, not a full NLP pipeline (proper tokenization, lemmatization, and stop-word lists belong in the dedicated text notebook). Sampled for speed.

In [ ]:
if HAS_ABSTRACTS:
    STOP = set("the a an and or of to in for with we were was is are be by on at as that this "
               "from been being has have had not but their there which study patients results "
               "methods conclusion conclusions background objective using used also between these "
               "than into more most can may such our both per via".split())
    cnt = collections.Counter()
    for txt in df['abstract'].fillna('').sample(min(50000, len(df)), random_state=0):
        for w in re.findall(r"[a-z]{4,}", txt.lower()):
            if w not in STOP:
                cnt[w] += 1
    topw = pd.Series(dict(cnt.most_common(20)))[::-1]
    plt.figure(figsize=(10, 6))
    sns.barplot(x=topw.values, y=topw.index, color="#8a5fb0")
    plt.title("Most frequent abstract words (sampled, stop-words removed)")
    plt.xlabel("occurrences"); plt.ylabel(""); plt.tight_layout(); plt.show()
else:
    print("abstract text absent (metadata-only export) — skipping vocabulary view.")

**What this shows:** a quick word-frequency view with a small stop-word filter — a rough sense of dominant vocabulary, not real NLP. Proper term analysis (stop-word lists, n-grams, TF-IDF, lemmatization) and any modelling are deliberately deferred to the topic-modelling and NER notebooks; this cell only confirms the text is usable and hints at what those will find.

### Note — MeSH top-level categories (not included)
A top-level category breakdown (Diseases, Chemicals, Anatomy, …) needs the actual MeSH tree numbers from NLM (the descriptor → tree-number table). The first-letter-of-the-name shortcut does not work — descriptor names do not start with their category letter (e.g. "Neoplasms" is category C, "Humans" is B). Left for a dedicated MeSH-analysis notebook where the tree file can be joined.

## 9. Summary, uses, and caveats

### What this analysis found
- **Volume** grows steadily from the mid-1990s to ~125k US-affiliated articles/year by 2025, excluding the artifact window.
- **The 2013–2015 dip is an indexing artifact** — established journals dropped out of the US-affiliation filter during PubMed's ~2013–2014 affiliation-indexing change, not a real decline. Treat pre-2014 volumes as lower bounds.
- **Team science dominates by 2025** — solo authorship fell 14.6%→2.5%; the 6–20 author band is now the majority.
- **Abstracts lengthened ~45%** (≈1,134→1,639 chars, 1994→2025).
- **MeSH depth was stable ~13 descriptors through 2019, then fell to ~8** (NLM automated indexing, not a content change).
- **Journals are broadly distributed** — top 100 ≈ 19% of output across 7,316 journals (Gini in 4e).
- **Keyword and COI fields are sparse before ~2012 and ~2017** respectively (auto-detected in 7a).

### What this dataset enables

**With the metadata alone (the published release)**
- **Bibliometrics & science-of-science** — publication-volume trends, journal turnover (§4c–d), team-size evolution (§5b), concentration analysis (§4e). Ready to use as-is.
- **MeSH-based topic mapping** — track topic prevalence over time with the normalized view (§6e), build MeSH co-occurrence graphs, or cluster articles by descriptor profile. Strip generic descriptors ("Humans", "Female", …) first.
- **Author & affiliation network analysis** — co-authorship graphs from `author_names`, institutional patterns from `affiliations`. Restrict to post-2014 for reliable affiliation coverage; names are not disambiguated.
- **Linkage spine** — every record carries its PMID, so this metadata joins cleanly to citation data (iCite, OpenCitations), funding data, or full text from PMC. It is a backbone to attach other sources to.
- **Sampling frame** — draw stratified samples (by year, journal, topic) for a downstream study, then fetch only what is needed.

**With abstracts (local clean corpus, or re-fetched by PMID)**
- **Text mining / NER** — entity extraction (diseases, genes, drugs) over ~3M abstracts.
- **Topic modelling** — LDA / BERTopic over abstract text, segmented by the eras this EDA identified.
- **Semantic search / embeddings** — embed abstracts for similarity search; the metadata supplies filters (year, journal, MeSH) to scope queries.
- **Trend & structure work** — abstract length and section-label trends are in §8; richer text analysis needs the full text.

### Highest-value next steps
1. Join PMIDs to **citation counts** → how team size, journal, or MeSH topic relate to impact.
2. **MeSH co-occurrence network** → the topical structure of US biomedical research and how it shifted over 30 years.
3. Re-fetch abstracts for a **topic-modelling** pass on a clean post-2015 slice (sidesteps the artifacts).

### Caveats for any use
- **2013–2015 volume** is an affiliation-indexing artifact — exclude or flag it for any longitudinal rate analysis; treat pre-2014 counts as lower bounds.
- **~77% of records have imprecise dates** — use `pubdate_precision`; restrict to `full_date` for month-level work.
- **No abstract text in the published release** — text methods need the local build or a re-fetch by PMID.
- **Author names are not disambiguated** — "J Smith" is not unified across records.
- **Keywords** valid from ~2012; early records contain non-biomedical noise (filter before use).
- **MeSH** depth changes after 2019 — segment by era; drop generic descriptors before clustering.
- **COVID-19** MeSH exists only from 2020 — union with "Coronavirus Infections", "SARS Virus", "Betacoronavirus" for the pre-2020 baseline.
- **Co-authorship networks** — restrict to post-2014; large consortia (max ~2,929 authors) create hub nodes.
- **Scope** — US-affiliated, human-subject, English-language only; this is not all of PubMed, so generalize accordingly.